In [11]:
import dolfinx
print(dolfinx.__version__)

0.11.0


In [12]:
from dolfinx import fem, io
import numpy as np
from mpi4py import MPI
import gmsh

In [13]:
# gmsh
gmsh.initialize()
gmsh.model.add("bar")

gmsh.model.occ.importShapes("./bar.step")
gmsh.model.occ.synchronize()

Info    :  - Label 'Shapes/Body' (3D)
Info    :  - Color (0.678431, 0.709804, 0.741176) (3D & Surfaces)


In [14]:
volumes = gmsh.model.getEntities(dim=3)
print("volume : ",volumes)

volume_tag = volumes[0][1]

boundary = gmsh.model.getBoundary(
    [(3,volume_tag)],
    oriented=False,
    recursive = False
)

surfaces = [tag for dim,tag in boundary if dim == 2]
print("surface ",surfaces)

volume :  [(3, 1)]
surface  [1, 2, 3, 4, 5, 6]


In [15]:
mass_properties = gmsh.model.occ.getMass(3, volume_tag)
# print("volume = ",mass_properties)

tol = 1e-5
p = [0.0, 0.0, 0.0]
left_face = [] # assume x,y,z = 0 xmax=xmin=0   
for surface_tag in surfaces:
    # area = gmsh.model.occ.getMass(2,surface_tag)
    # print("surface ", surface_tag, " area = ",area)

    xmin,ymin,zmin,xmax,ymax,zmax = gmsh.model.getBoundingBox(2,surface_tag)
  
    if (xmin - tol <= p[0] <= xmax+tol) and (ymin - tol <= p[1] <= ymax+ tol) and (zmin - tol <= p[2] <= zmax + tol ):
        left_face.append(surface_tag)

print(left_face)

[5]


In [16]:
# create physical group
gmsh.model.addPhysicalGroup(
    3,[1],1,name="bar"
)

gmsh.model.addPhysicalGroup(
    2,left_face,2,name="fixed_left_face"
)

2

In [17]:
# generating mesh
gmsh.option.setNumber("Mesh.MeshSizeMin", 0.1)
gmsh.option.setNumber("Mesh.MeshSizeMax", 1.0)
gmsh.model.mesh.generate(3)

# from gmsh to dolfinx
mesh_data = io.gmsh.model_to_mesh(
    gmsh.model,
    MPI.COMM_WORLD,
    0,
    gdim=3
)

mesh = mesh_data.mesh
cell_tags = mesh_data.cell_tags
facet_tags =  mesh_data.facet_tags

gmsh.write("bar1.msh")
gmsh.finalize()

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 10%] Meshing curve 2 (Line)
Info    : [ 20%] Meshing curve 3 (Line)
Info    : [ 30%] Meshing curve 4 (Line)
Info    : [ 40%] Meshing curve 5 (Line)
Info    : [ 50%] Meshing curve 6 (Line)
Info    : [ 60%] Meshing curve 7 (Line)
Info    : [ 60%] Meshing curve 8 (Line)
Info    : [ 70%] Meshing curve 9 (Line)
Info    : [ 80%] Meshing curve 10 (Line)
Info    : [ 90%] Meshing curve 11 (Line)
Info    : [100%] Meshing curve 12 (Line)
Info    : Done meshing 1D (Wall 0.0026166s, CPU 0s)
Info    : Meshing 2D...
Info    : [  0%] Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : [ 20%] Meshing surface 2 (Plane, Frontal-Delaunay)
Info    : [ 40%] Meshing surface 3 (Plane, Frontal-Delaunay)
Info    : [ 60%] Meshing surface 4 (Plane, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 5 (Plane, Frontal-Delaunay)
Info    : [ 90%] Meshing surface 6 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.501579s, CPU 0.45

In [18]:
import ufl

V = fem.functionspace(mesh,("Lagrange",1,(mesh.geometry.dim,)))

E = 70e9
nu = 0.3
rho = 2700

mu = E / (2* (1+nu))
lambda_ = E * nu / (
    (1 + nu) * (1 - 2 * nu)
)

def epsilon(u):
    return ufl.sym(ufl.grad(u)) 

def sigma(u):
    return (
        lambda_ * ufl.tr(epsilon(u)) * ufl.Identity(3)
        + 2 * mu * epsilon(u)
    )   

In [19]:
u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

a = ufl.inner(
    sigma(u),
    epsilon(v)
) * ufl.dx

m = rho * ufl.inner(u, v) * ufl.dx

fixed_facets = facet_tags.find(2)
fixed_dofs = fem.locate_dofs_topological(
    V,
    mesh.topology.dim - 1,
    fixed_facets
)

u_zero = fem.Constant(
    mesh,
    (0.0, 0.0, 0.0)
)

bc = fem.dirichletbc(
    u_zero,
    fixed_dofs,
    V
)

from dolfinx.fem import petsc

K = petsc.assemble_matrix(
    fem.form(a),
    bcs=[bc]
)

K.assemble()


M = petsc.assemble_matrix(
    fem.form(m),
    bcs=[bc]
)

M.assemble()

In [ ]:
# # solving the eigenvalue problem
from slepc4py import SLEPc

eps = SLEPc.EPS().create(mesh.comm)

eps.setOperators(K, M)

eps.setProblemType(
    SLEPc.EPS.ProblemType.GHEP
)

eps.setWhichEigenpairs(
    SLEPc.EPS.Which.SMALLEST_REAL
)

eps.setDimensions(nev=10)

eps.solve()


nconv = eps.getConverged()

print("Converged eigenvalues:", nconv)


for i in range(nconv):

    eigenvalue = eps.getEigenvalue(i)

    omega = eigenvalue ** 0.5

    frequency = omega / (2 * 3.141592653589793)

    print(
        "Mode:",
        i + 1,
        "frequency:",
        frequency,
        "Hz"
    )




with io.XDMFFile(mesh.comm, "modes1.xdmf", "w") as file:
    file.write_mesh(mesh)

    for i in range(nconv):

        # Create function for this mode
        mode = fem.Function(V)
        mode.name = f"Mode_{i+1}"

        # Get eigenvector
        eps.getEigenvector(i, mode.x.petsc_vec)
        mode.x.scatter_forward()

        # Normalize only for visualization
        max_value = np.max(np.abs(mode.x.array))
        if max_value > 0:
            mode.x.array[:] /= max_value

        # Frequency
        eigenvalue = eps.getEigenvalue(i)
        omega = np.sqrt(eigenvalue.real)
        frequency = omega / (2*np.pi)

        print(f"Mode {i+1}: {frequency:.4f} Hz")

        # Save mode
        file.write_function(mode)